In [53]:
from openai import OpenAI
from bs4 import BeautifulSoup
import requests
import json
from IPython.display import Markdown, display, update_display

In [54]:
ollama_baseurl = "http://localhost:11434/v1"

ollama = OpenAI(base_url=ollama_baseurl,api_key='ollama')

def ollama_fun(prompt,model):
    response = ollama.chat.completions.create(
        model=model,
        messages=[{
            "role":"user",
            "content":prompt
        }]
    )
    return response.choices[0].message.content
    

In [55]:
headers ={
        "User-Agent":"Chrome/136.0.0.0"
        }

def fetchUrl(url):

    response = requests.get(url,headers=headers)
    if response.status_code == 200:
        print("Successfully Fetched!")
        soup = BeautifulSoup(response.text,"html.parser")
        links = [link.get('href') for link in soup.find_all('a')]

    return [link for link in links if link]


In [102]:

fetchUrl("https://www.langchain.com/")

Successfully Fetched!


['https://www.langchain.com/blog/interrupt-2026-overview',
 '/',
 '/langsmith-platform',
 '/langsmith/observability',
 '/langsmith/evaluation',
 '/langsmith/deployment',
 '/langsmith/fleet',
 '/langsmith/sandboxes',
 '/deep-agents',
 '/langchain',
 '/langgraph',
 '/blog',
 '/customers',
 '/resources',
 'https://www.youtube.com/playlist?list=PLfaIDFEXuae3UwB1QGEjsRAr8BzCQss7s',
 'https://academy.langchain.com/',
 'https://www.youtube.com/@LangChain',
 'https://docs.langchain.com/',
 '/startups',
 'https://luma.com/langchain?k=c',
 '/community',
 'https://docs.langchain.com/',
 '/about',
 '/careers',
 '/langchain-partner-network',
 '/events',
 '/pricing',
 'https://smith.langchain.com/',
 '/contact-sales',
 'https://smith.langchain.com/',
 '/contact-sales',
 'https://smith.langchain.com/',
 '/contact-sales',
 '#langsmith-engine',
 '#observability',
 '#evaluation',
 '#deployment',
 '#fleet',
 'https://www.langchain.com/blog/introducing-langsmith-engine',
 '/langsmith/observability',
 '/la

In [57]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [87]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the absolute full https URL in JSON format.
Do not include Terms of Service, Privacy, email links,broken links

Links (some might be relative links):

"""
    links = fetchUrl(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [103]:
print(get_links_user_prompt("https://www.langchain.com/"))

Successfully Fetched!

Here is the list of links on the website https://www.langchain.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the absolute full https URL in JSON format.
Do not include Terms of Service, Privacy, email links,broken links

Links (some might be relative links):

https://www.langchain.com/blog/interrupt-2026-overview
/
/langsmith-platform
/langsmith/observability
/langsmith/evaluation
/langsmith/deployment
/langsmith/fleet
/langsmith/sandboxes
/deep-agents
/langchain
/langgraph
/blog
/customers
/resources
https://www.youtube.com/playlist?list=PLfaIDFEXuae3UwB1QGEjsRAr8BzCQss7s
https://academy.langchain.com/
https://www.youtube.com/@LangChain
https://docs.langchain.com/
/startups
https://luma.com/langchain?k=c
/community
https://docs.langchain.com/
/about
/careers
/langchain-partner-network
/events
/pricing
https://smith.langchain.com/
/contact-sales
https://smith.langchain.com/
/contact-sales
https://smith.

In [104]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [106]:
select_relevant_links("https://www.langchain.com/")

Successfully Fetched!


{'links': [{'type': 'About page', 'url': 'https://langchain.com/about'},
  {'type': 'Careers/Jobs page', 'url': 'https://langchain.com/careers'},
  {'type': 'Blog', 'url': 'https://blog.langchain.com/'},
  {'type': 'Community & Join Community',
   'url': 'https://academy.langchain.com/join-community'},
  {'type': 'Resources', 'url': 'https://docs.langchain.com/'},
  {'type': 'FAQ/Support', 'url': 'https://support.langchain.com/'},
  {'type': 'Trust & Trust URL', 'url': 'https://trust.langchain.com/'}]}

In [107]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [108]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [109]:
print(fetch_page_and_all_relevant_links("https://www.langchain.com/"))

Successfully Fetched!
## Landing Page:

LangChain: Observe, Evaluate, and Deploy Reliable AI Agents

Catch up on everything we shipped at Interrupt
Explore the launches
Products
LangSmith Platform
Observability
See exactly what your agents are doing
Evaluation
Score and improve agent performance
Deployment
Ship and scale agents in production
Fleet
Agents for the whole company
Sandboxes
Run agent-generated code safely
Open Source Frameworks
deepagents
Build long-running agents for complex tasks
langchain
Quick start agents with any model provider
langgraph
Build reliable agents with low-level control
Learn
Resources
Blog
Customer Stories
Guides
Max Agency
How-To
LangChain Academy
YouTube
Documentation
Community
LangSmith for Startups
Meetups
Community
Docs
Company
About
Careers
Partners
Events
Pricing
Try LangSmith
Get a demo
Try LangSmith
Get a demo
Powering the
Agent Development Lifecycle
Make experimentation repeatable, iterate faster, and gain momentum with LangSmith.
Start building

In [97]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [98]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [110]:
get_brochure_user_prompt("Langchain", "https://www.langchain.com/")

Successfully Fetched!


"\nYou are looking at a company called: Langchain\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nLangChain: Observe, Evaluate, and Deploy Reliable AI Agents\n\nCatch up on everything we shipped at Interrupt\nExplore the launches\nProducts\nLangSmith Platform\nObservability\nSee exactly what your agents are doing\nEvaluation\nScore and improve agent performance\nDeployment\nShip and scale agents in production\nFleet\nAgents for the whole company\nSandboxes\nRun agent-generated code safely\nOpen Source Frameworks\ndeepagents\nBuild long-running agents for complex tasks\nlangchain\nQuick start agents with any model provider\nlanggraph\nBuild reliable agents with low-level control\nLearn\nResources\nBlog\nCustomer Stories\nGuides\nMax Agency\nHow-To\nLangChain Academy\nYouTube\nDocumentation\nCommunity\nLangSmith for Startups\nMeetups\nCommunity\nDo

In [45]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [111]:
create_brochure("LangChain", "https://www.langchain.com/")

Successfully Fetched!


# Welcome to LangChain: Building the Future of Agents

At LangChain, we're passionate about creating the tools and platforms that enable companies to ship great agents faster. Our mission is to unlock the full potential of Large Language Models (LLMs) by building them into reliable agents that can use data and take actions.

## Our Story
LangChain was born out of a passion for agent development from our founder, Harrison Chase. When ChatGPT launched in late 2022, everything changed. We saw an opportunity to create a platform that would help companies overcome the challenges of building reliable agents. With the help of co-founder Ankush Gola, we started LangChain in early 2023 and have been building ahead of the industry ever since.

## Our Products

* **LangSmith Platform**: Our flagship product that powers the agent development lifecycle. It provides observeability, evaluation, deployment, and more features to make experimentation repeatable and iterate faster.
* **langchain**: An open-source framework for quick-start agents with any model provider.
* **langgraph**: A framework for building reliable agents with low-level control.

## Our Values

At LangChain, we value innovation, collaboration, and customer success. We believe that LLMs are incredibly powerful when put to work through agents that can use data and take actions.

## Customer Stories
We're proud to serve top AI teams from startups to global enterprises, powering their agent development needs with our platform.

## Join Our Team
Are you passionate about building the future of agents? We're hiring talented engineers, researchers, and product specialists to join our team. Check out our [Careers page](link) to learn more.

## Get in Touch
Want to learn more about LangChain or get started with our products? Contact us at [info@langchain.com](mailto:info@langchain.com) or schedule a demo today.

## Stay Ahead of the Curve

Subscribe to our blog and YouTube channel for insights on agent development, AI, and innovation.

In [113]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [114]:
stream_brochure("LangChain","https://www.langchain.com/")

Successfully Fetched!


# LangChain: Building Reliable AI Agents for a Better Future

Empowering the world's top AI teams to develop trustworthy agents with LangChain.

At LangChain, we believe that the key to unlocking the true potential of Artificial Intelligence lies in reliable agent development. Our platform, LangSmith, is designed to simplify the process of building and deploying high-performance AI models, empowering users to focus on what matters most: maximizing impact.

## What Sets Us Apart

We're not just another AI company; we're a community-driven movement pushing the boundaries of what's possible with LangChain. With a strong focus on open-source frameworks and cutting-edge technology, we provide:

*   **Fleet:** A platform that enables businesses to create helpful AI agents without requiring code expertise.
*   **LangSmith Engine**: A comprehensive toolset for surface-level analysis and issue resolution, streamlining the agent development lifecycle.

## Our Mission

By providing a robust foundation for agent development, LangChain empowers organizations to:

*   **Build** more efficient AI models
*   **Test** and **evaluate** their performance
*   **Deploy** agents with confidence
*   **Iterate faster**, leading to accelerated momentum and real-world impact.

## The Benefits

LangChain's innovative approach brings numerous benefits, including:

*   **Improved reliability**: By streamlining agent development and deployment, our platform minimizes errors and maximizes results.
*   **Enhanced collaboration**: With Fleet, teams can work together more efficiently, ensuring that everyone works towards the same goal.
*   **Accelerated innovation**: The LangSmith Engine helps identify and resolve issues faster, saving time and improving product quality.

## Join Us

Ready to be part of a new wave of AI innovation? Explore our resources below:

### Get Started with LangChain
Click the link to start your journey today! [Lang Chain Documentation](https://docs.langchain.com/)
### Learn More About Our Products
Read our blog, review customer stories, and stay updated on our latest releases: [Lang Chain Blog](https://blog.langchain.org/)

Stay up-to-date with LangChain's mission. Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain.
Buy tickets >